code example...using a votingclassifier but using different columns for each classifier...

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import VotingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

rf_features = ["age", "income", "debt"]
lr_features = ["tfidf_1", "tfidf_2"]
svm_features = ["login_count", "days_since_login"]

rf_pipe = Pipeline([
    ("select_columns", ColumnTransformer([
        ("rf_cols", "passthrough", rf_features)
    ])),
    ("classifier", RandomForestClassifier(random_state=42))
])

lr_pipe = Pipeline([
    ("select_columns", ColumnTransformer([
        ("lr_cols", "passthrough", lr_features)
    ])),
    ("classifier", LogisticRegression(max_iter=1000))
])

svm_pipe = Pipeline([
    ("select_columns", ColumnTransformer([
        ("svm_cols", "passthrough", svm_features)
    ])),
    ("classifier", SVC(probability=True, random_state=42))
])

voter = VotingClassifier(
    estimators=[
        ("rf", rf_pipe),
        ("lr", lr_pipe),
        ("svm", svm_pipe)
    ],
    voting="soft"
)

voter.fit(X_train, y_train)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
from sklearn.ensemble import VotingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

numeric_features = ["age", "income", "debt"]
text_features = ["tfidf_score_1", "tfidf_score_2", "tfidf_score_3"]
behaviour_features = ["login_count", "days_since_last_login"]

rf_pipe = Pipeline([
    ("select_features", ColumnTransformer([
        ("num", "passthrough", numeric_features)
    ])),
    ("model", RandomForestClassifier(random_state=42))
])

lr_pipe = Pipeline([
    ("select_features", ColumnTransformer([
        ("text", "passthrough", text_features)
    ])),
    ("model", LogisticRegression(max_iter=1000))
])

svm_pipe = Pipeline([
    ("select_features", ColumnTransformer([
        ("behaviour", "passthrough", behaviour_features)
    ])),
    ("model", SVC(probability=True, random_state=42))
])

voter = VotingClassifier(
    estimators=[
        ("rf", rf_pipe),
        ("lr", lr_pipe),
        ("svm", svm_pipe),
    ],
    voting="soft"
)

voter.fit(X_train, y_train)

1. Tune each base model recipe separately.
2. Wrap each tuned model recipe in CalibratedClassifierCV.
3. Put those calibrated wrappers into VotingClassifier(voting="soft").
4. Fit the voting classifier on training data.
5. Evaluate once on untouched test data.

In [ ]:
rf_best = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    random_state=42
)

svm_best = LinearSVC(
    C=0.5,
    random_state=42
)

sgd_best = SGDClassifier(
    loss="hinge",
    alpha=0.0001,
    random_state=42
)

rf_cal = CalibratedClassifierCV(rf_best, method="sigmoid", cv=cv, ensemble=False)
svm_cal = CalibratedClassifierCV(svm_best, method="sigmoid", cv=cv, ensemble=False)
sgd_cal = CalibratedClassifierCV(sgd_best, method="sigmoid", cv=cv, ensemble=False)

voter = VotingClassifier(
    estimators=[
        ("rf", rf_cal),
        ("svm", svm_cal),
        ("sgd", sgd_cal),
    ],
    voting="soft",
    weights=[1, 1, 1]
)

voter.fit(X_train, y_train)

In [ ]:
from sklearn.ensemble import VotingClassifier, RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.linear_model import SGDClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf_cal = CalibratedClassifierCV(
    estimator=RandomForestClassifier(random_state=42),
    method="sigmoid",
    cv=cv,
    ensemble=False
)

svm_cal = CalibratedClassifierCV(
    estimator=LinearSVC(random_state=42),
    method="sigmoid",
    cv=cv,
    ensemble=False
)

sgd_cal = CalibratedClassifierCV(
    estimator=SGDClassifier(loss="hinge", random_state=42),
    method="sigmoid",
    cv=cv,
    ensemble=False
)

voter = VotingClassifier(
    estimators=[
        ("rf", rf_cal),
        ("svm", svm_cal),
        ("sgd", sgd_cal),
    ],
    voting="soft"
)

voter.fit(X_train, y_train)